In [1]:
import AILibs
import numpy

dataset_root_path = "/users/michal/datasets/UWaveGestureLibrary/"

dataset_train   = AILibs.datasets.UWaveGestureDataset(dataset_root_path, split='TRAIN')
dataset_test    = AILibs.datasets.UWaveGestureDataset(dataset_root_path, split='TEST')

x_train = dataset_train.features
y_train = dataset_train.labels

x_test = dataset_test.features
y_test = dataset_test.labels

num_classes = dataset_train.num_classes

print("Extracting features")
#features_extractor = AILibs.features.Catch22Features(x_train)
features_extractor = AILibs.features.RocketFeatures(x_train, 1024)

# obtain features for train and test sets
z_train = features_extractor.forward(x_train)
z_test  = features_extractor.forward(x_test)

#z_train = numpy.reshape(x_train, (x_train.shape[0], -1))
#z_test  = numpy.reshape(x_test, (x_test.shape[0], -1))

print("Feature shape: ", z_train.shape)
print("Test feature shape: ", z_test.shape)

print("Training forest")

forest = AILibs.forest.RandomForest()   


# one hot encoding
y_one_hot = numpy.eye(num_classes)[y_train.astype(int)]

#forest.fit(z_train, y_one_hot, max_depth=8, num_trees=256, num_subsamples=-1, num_random_candidates=16)
forest.fit(z_train, y_one_hot, max_depth=10, num_trees=256, num_subsamples=-1, num_random_candidates=16)



Loading TRAIN dimensions...
Loaded 120 samples.
Feature shape per sample: (315, 3) (seq_length, num_features)
Loading TEST dimensions...
Loaded 320 samples.
Feature shape per sample: (315, 3) (seq_length, num_features)
Extracting features
Feature shape:  (120, 3024)
Test feature shape:  (320, 3024)
Training forest


In [2]:


print("Predicting with Random Forest...")
y_pred = forest.predict_batch(z_test)


metrics = AILibs.metrics.classification_evaluation(y_test, y_pred,num_classes)


for key, value in metrics.items():
    print(f"{key}: {value}")

    

Predicting with Random Forest...
n_samples: 320
num_classes: 8
accuracy: 0.925
macro_precision: 0.92796
macro_recall: 0.925
macro_f1_score: 0.92444
macro_mcc: 0.91508
macro_specificity: 0.98929
macro_balanced_accuracy: 0.95714
macro_iou: 0.86154
macro_dice: 0.92444
tp_per_class: [39, 39, 35, 35, 40, 35, 33, 40]
tn_per_class: [272, 276, 277, 279, 278, 279, 277, 278]
fp_per_class: [8, 4, 3, 1, 2, 1, 3, 2]
fn_per_class: [1, 1, 5, 5, 0, 5, 7, 0]
precision_per_class: [0.82979, 0.90698, 0.92105, 0.97222, 0.95238, 0.97222, 0.91667, 0.95238]
recall_per_class: [0.975, 0.975, 0.875, 0.875, 1.0, 0.875, 0.825, 1.0]
f1_score_per_class: [0.89655, 0.93976, 0.89744, 0.92105, 0.97561, 0.92105, 0.86842, 0.97561]
